# DeepRacer-Genesis on Colab

AWS DeepRacer RL environment ported to [Genesis](https://github.com/Genesis-Embodied-AI/Genesis) — ROS-free, GPU-batched, trained with rsl-rl PPO.

This notebook installs the package, trains a driving policy with PPO on the GPU, and renders a bird's-eye video of all the agents driving at once.

**Runtime**: needs a GPU runtime (T4 works). Note that Colab VMs expose only the CUDA *compute* driver — no NVIDIA Vulkan/EGL graphics stack — so the Madrona/Nyx **camera-observation renderers cannot run here**; physics + training run on CUDA and the demo video renders through Mesa (software). For vision-based training use a machine with the full NVIDIA driver.

Each phase runs as a subprocess because Genesis initializes once per process.

In [ ]:
# @title GPU check
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

In [ ]:
# @title Install deepracer-genesis
REPO = "https://github.com/Luna-v0/deepracer-genesis.git"
BRANCH = "main"

# Colab preinstalls TensorFlow, whose bundled LLVM symbols clash with Mesa's
# software renderer (segfault at first render once rsl-rl imports
# tensorboard). We don't use TF -> remove it.
!pip uninstall -q -y tensorflow tensorflow-cpu tf-keras 2>/dev/null

# --force-reinstall --no-cache-dir: after a "Restart runtime" the VM disk (and
# any previously installed/broken build of this package) survives — never
# trust a cached install when switching branches.
import glob
local_wheel = sorted(glob.glob("/content/deepracer_genesis-*.whl"))
if local_wheel:
    print("installing local wheel:", local_wheel[-1])
    !pip install -q --force-reinstall --no-deps {local_wheel[-1]}
    !pip install -q {local_wheel[-1]}
else:
    !pip install -q --no-cache-dir --force-reinstall --no-deps "deepracer-genesis @ git+{REPO}@{BRANCH}"
    !pip install -q "deepracer-genesis @ git+{REPO}@{BRANCH}"

# fail fast if the install is unusable
import subprocess, sys
r = subprocess.run([sys.executable, "-c",
    "from deepracer_genesis.configs.cfgs import get_env_cfg; "
    "from deepracer_genesis import ASSETS_DIR; import os; "
    "assert os.path.exists(os.path.join(ASSETS_DIR, 'routes', 'reinvent_base.npy')), 'assets missing'; "
    "print('install OK: subpackages + assets present')"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode != 0:
    raise RuntimeError(
        "deepracer-genesis installed but is incomplete. Use a branch that "
        "includes the packaging fixes (all branches on Luna-v0/deepracer-genesis "
        "do since v0.1.1), or upload a wheel built with `uv build --wheel`.")

In [ ]:
%%writefile /content/smoke.py
# smoke test: the batched physics env builds and steps (cell magic must be line 1)
import time
import torch
import genesis as gs
gs.init(backend=gs.cuda, logging_level="warning")
from deepracer_genesis.configs.cfgs import get_env_cfg
from deepracer_genesis.envs import DeepRacerEnv
env = DeepRacerEnv(num_envs=64, env_cfg=get_env_cfg(vision=False))
a = torch.zeros(64, 2, device=env.device)
for _ in range(20):
    env.step(a)
t0 = time.perf_counter()
for _ in range(200):
    env.step(a)
print("physics OK:", round(64 * 200 / (time.perf_counter() - t0)), "steps/s aggregate")

In [ ]:
# first run includes Genesis JIT compilation (a few minutes on a T4)
!python /content/smoke.py

In [ ]:
# @title Train a driving policy (state-based PPO, with per-env domain randomization)
# ~10 min on a T4. Bump --max_iterations for confident lapping (300+).
!cd /content && python -m deepracer_genesis.train -B 512 --randomize --max_iterations 60 --exp_name colab 2>&1 | grep -E "Mean reward|episode length" | tail -6

In [ ]:
# @title Render the many-agents demo video (bird's-eye, all cars at once)
# Colab renders this through Mesa (software): ~0.4 s/frame at 640x480 after a
# one-time ~35 s pipeline warmup, so 400 frames takes a few minutes.
import glob
ckpts = sorted(glob.glob("/content/logs/colab/model_*.pt"),
               key=lambda p: int(p.rsplit("_", 1)[1][:-3]))
assert ckpts, ("No checkpoint found in /content/logs/colab/ — the training "
               "cell above must complete first (rerun it and check its output).")
ckpt = ckpts[-1]
print("checkpoint:", ckpt)
!cd /content && python -m deepracer_genesis.eval --checkpoint {ckpt} --num_envs 16 --steps 400 --res 640x480 --out logs/eval 2>&1 | tail -3

In [ ]:
# @title Watch it
from IPython.display import Video
Video("/content/logs/eval/spectator.mp4", embed=True, width=720)

## Going further (on machines with a full NVIDIA driver)

Colab lacks the NVIDIA Vulkan/EGL graphics stack, so the camera-observation
renderers are unavailable here. On a workstation/server with the full driver:

- **Vision policy** (CNN on the 160x120 onboard camera, Madrona batch renderer):
  `python -m deepracer_genesis.train -B 128 --vision --max_iterations 300`
  (if the system CUDA toolkit is newer than 12.4, first run
  `scripts/fix_madrona_cuda13.sh` — aligns gs-madrona's bundled nvJitLink
  with a matching NVRTC)
- **Camera validation** (paired onboard + top-down images, 4 automated checks):
  `python -m deepracer_genesis.validation.camera_check --num_envs 4`
- **Heterogeneous tracks** (each env simulates a different track): pass
  `--tracks reinvent_base,reInvent2019_track,2022_reinvent_champ`
- **Domain randomization** follows the
  [Genesis DR guide](https://genesis-world.readthedocs.io/en/latest/user_guide/getting_started/policy_training/best_practices/domain_randomization.html):
  per-env friction/mass/COM per link, kp/kv/armature per dof (`--randomize`).
- **Throughput table**: `python benchmarks/throughput.py --sweep` (clone the repo).
- Branches: `raster-vision` (color-faithful rasterizer observations),
  `nyx-vision` (photorealistic Nyx renderer, driver 575+), `cpu-vision` (no GPU).